In [ ]:
import h5py
import numpy as np
import matplotlib.pyplot as plt
import os
from pathlib import Path

# ================= 配置区域 =================
# 1. 您的数据路径 (保持不变)
DATA_DIR = "/home/jovyan/gpu_space/workspace_jiayi/alex_datasets/3D_minimal"

# 2. 输出文件夹 (自动创建，用于存放生成的素材)
OUTPUT_DIR = "fig2_3_candidates"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# 3. 通道定义 (基于论文 background.tex 推断)
# -------------------------------------------------
# 0-14: QTI Scalars (uFA is usually last, index 14)
# 225-229: CEST Params (Amide usually first, index 225)
# 341: MPRAGE (Reference)
# 342-350: QSM/SMWI (我们暂选 342，如果全黑或噪点多，可尝试 343, 344)
# -------------------------------------------------
CH_UFA    = 14
CH_AMIDE  = 225
CH_MPRAGE = 341
CH_QSM    = 342  # <--- 如果图像看起来不对，尝试修改为 343 或 344

# 4. 切片范围 (挑选解剖结构最丰富的一段)
AXIS = 'Axial'
START_SLICE = 135
END_SLICE   = 145
STEP        = 1      # 每一层都画，不错过细节

# ================= 工具函数 =================
def normalize(img, p_min=1, p_max=99):
    """鲁棒归一化，确保图像对比度适宜"""
    if np.all(img == 0): return img
    vmin, vmax = np.percentile(img, [p_min, p_max])
    return np.clip(img, vmin, vmax)

def get_slice(data, axis, idx):
    """提取切片并旋转"""
    if axis == 'Axial':   # Z-axis
        return np.rot90(data[:, :, idx, :])
    elif axis == 'Coronal': # Y-axis
        return np.rot90(data[:, idx, :, :])
    else: # Sagittal
        return np.rot90(data[idx, :, :, :])

# ================= 主逻辑 =================

# 1. 加载数据
mat_files = sorted(list(Path(DATA_DIR).glob('*.mat')))
if not mat_files:
    raise FileNotFoundError(f"在 {DATA_DIR} 未找到 .mat 文件")

target_file = mat_files[0]
print(f"📂 正在读取: {target_file.name} ...")

with h5py.File(target_file, 'r') as f:
    # 假设数据格式为 (C, X, Y, Z)，转置为 (X, Y, Z, C)
    raw_data = f['data'][:]
    data_vol = np.moveaxis(raw_data, 0, -1)
    print(f"✅ 数据加载完成。Shape: {data_vol.shape}")


In [ ]:

# 2. 循环生成对比图
print(f"🚀 开始生成候选切片 (保存至 {OUTPUT_DIR}/)...")

for slice_idx in range(START_SLICE, END_SLICE, STEP):
    try:
        sl_data = get_slice(data_vol, AXIS, slice_idx)
    except IndexError:
        continue

    # 提取四个模态
    img_mprage = sl_data[..., CH_MPRAGE]
    img_ufa    = sl_data[..., CH_UFA]
    img_amide  = sl_data[..., CH_AMIDE]
    img_qsm    = sl_data[..., CH_QSM]

    # 创建画布 (1行4列)
    fig, axes = plt.subplots(1, 4, figsize=(20, 5), dpi=150)
    fig.suptitle(f"Figure 2.3 Asset Candidate: Slice {slice_idx}", fontsize=16)
    
    # 统一绘图风格
    contrasts = [
        (img_mprage, 'gray',   'MPRAGE (Structural)'),
        (img_ufa,    'magma',  'QTI (uFA)'),
        (img_amide,  'inferno','CEST (Amide)'),
        (img_qsm,    'bone',   f'QSM/SMWI (Ch{CH_QSM})')
    ]
    
    for ax, (img, cmap, title) in zip(axes, contrasts):
        # 使用 Nearest 插值以保留像素原本的锯齿感 (体现分辨率差异)
        # 或者使用 bicubic 看起来更平滑？ -> 论文图通常建议 Nearest 或 output 尺寸够大
        ax.imshow(normalize(img), cmap=cmap, aspect='equal', interpolation='nearest')
        ax.set_title(title, fontsize=12)
        ax.axis('off')

    # 保存文件
    save_path = os.path.join(OUTPUT_DIR, f"slice_{slice_idx}_comparison.png")
    plt.savefig(save_path, bbox_inches='tight')
    
    # 在 Notebook 中显示 (只显示前2张测试，避免刷屏)
    if slice_idx < START_SLICE + 20:
        plt.show()
    else:
        plt.close(fig) # 后面的只保存不显示

print(f"\n✅ 所有图片已生成。请查看 {OUTPUT_DIR} 文件夹挑选最佳切片。")